# Lesson 7 — Human-in-the-Loop (HITL)

## What you will learn
- `interrupt()` — pause the graph and wait for a human
- `MemorySaver` — **required** for interrupt to work (saves graph state)
- `thread_id` — identifies which conversation to resume
- `Command(resume=...)` — resume the graph after human input
- **3 real-world HITL patterns**

## Why HITL?
Agents can make mistakes. HITL lets you put humans at critical checkpoints:
- Before **dangerous actions** (delete, send email, spend money)
- When **information is missing** (agent needs user context)
- For **quality review** of generated content

## The Interrupt Mechanism
```
graph.invoke() → runs until interrupt() → PAUSES
                                             ↓
                                    human provides input
                                             ↓
graph.invoke(Command(resume=answer)) → CONTINUES from pause point
```

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

llm = ChatOllama(model="llama3.2", temperature=0)

---
# Pattern A — Approve/Reject Before Tool Execution

**Use case:** Agent wants to call a sensitive tool (send email, delete record).  
Before executing, pause and ask the human: **"Is this okay?"**

```
START → agent → human_approval ←interrupt→ [human says yes/no]
                     ↓ yes                    ↓ no
                   tools → agent → END      END (rejected)
```

In [ ]:
@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send an email. Sensitive action — requires human approval."""
    print(f"  ✉️  Email sent to {recipient}: '{subject}'")
    return f"Email sent to {recipient}"


class StateA(TypedDict):
    messages: Annotated[list, add_messages]


llm_a = llm.bind_tools([send_email])


def agent_a(state: StateA) -> dict:
    return {"messages": [llm_a.invoke([SystemMessage(content="Use tools when needed.")] + state["messages"])]}


def approval_node(state: StateA) -> dict:
    """Pause here and show the human what the agent wants to do."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        tc = last.tool_calls[0]
        print(f"\n⚠️  Agent wants to call: {tc['name']}")
        print(f"   Args: {tc['args']}")

        # INTERRUPT — execution pauses HERE
        decision = interrupt({
            "message": f"Agent wants to call '{tc['name']}'. Approve?",
            "tool": tc["name"],
            "args": tc["args"]
        })

        if str(decision).lower() != "yes":
            return {"messages": [AIMessage(content=f"Action '{tc['name']}' was rejected by user.")]}
    return {}


def route_a(state: StateA) -> str:
    last = state["messages"][-1]
    return "tools" if (hasattr(last, "tool_calls") and last.tool_calls) else "end"


b_a = StateGraph(StateA)
b_a.add_node("agent",    agent_a)
b_a.add_node("approval", approval_node)
b_a.add_node("tools",    ToolNode([send_email]))
b_a.add_edge(START,      "agent")
b_a.add_conditional_edges("agent",
    lambda s: "approval" if (hasattr(s["messages"][-1], "tool_calls") and s["messages"][-1].tool_calls) else "end",
    {"approval": "approval", "end": END})
b_a.add_conditional_edges("approval", route_a, {"tools": "tools", "end": END})
b_a.add_edge("tools", "agent")

ckpt_a  = MemorySaver()
graph_a = b_a.compile(checkpointer=ckpt_a)
print("Graph A compiled!")

In [ ]:
config_a = {"configurable": {"thread_id": "hitl-a-1"}}

# Step 1: Run — graph will pause at interrupt
result = graph_a.invoke(
    {"messages": [HumanMessage(content="Send an email to boss@company.com with subject 'Status Update' and body 'Project is on track.'")]},
    config=config_a
)

state = graph_a.get_state(config_a)
print(f"Graph paused: {bool(state.next)}")
if state.tasks:
    for task in state.tasks:
        if task.interrupts:
            print(f"Interrupt message: {task.interrupts[0].value['message']}")

In [ ]:
# Step 2: Resume with human's answer
# Change "yes" to "no" to test rejection
human_answer = "yes"   # <-- change this to "no" to reject

result = graph_a.invoke(Command(resume=human_answer), config=config_a)
print(f"Final: {result['messages'][-1].content}")

---
# Pattern B — Human Provides Missing Information

**Use case:** Agent needs user data it doesn't have.  
It pauses and asks the human directly.

```
START → check (no info?) → collect_info ←interrupt→ human provides data
                                ↓
                             agent (with info) → END
```

In [ ]:
class StateB(TypedDict):
    messages:  Annotated[list, add_messages]
    user_info: dict


def collect_info_node(state: StateB) -> dict:
    print("\n⚠️  Agent needs more information...")
    # INTERRUPT — ask human for missing data
    data = interrupt({"question": "Please provide: your name and preferred programming language"})
    return {"user_info": data}  # data comes back when resumed


def personalized_agent(state: StateB) -> dict:
    info = state.get("user_info", {})
    system = SystemMessage(content=f"Personalized context: {info}. Use it to tailor your response.")
    resp = llm.invoke([system] + state["messages"])
    return {"messages": [resp]}


b_b = StateGraph(StateB)
b_b.add_node("collect", collect_info_node)
b_b.add_node("agent",   personalized_agent)
b_b.add_conditional_edges(START,
    lambda s: "collect" if not s.get("user_info") else "agent",
    {"collect": "collect", "agent": "agent"})
b_b.add_edge("collect", "agent")
b_b.add_edge("agent", END)

ckpt_b  = MemorySaver()
graph_b = b_b.compile(checkpointer=ckpt_b)
print("Graph B compiled!")

In [ ]:
config_b = {"configurable": {"thread_id": "hitl-b-1"}}

# Run — will interrupt to collect info
graph_b.invoke({"messages": [HumanMessage(content="Give me a personalized learning tip.")], "user_info": {}}, config=config_b)

state = graph_b.get_state(config_b)
if state.tasks and state.tasks[0].interrupts:
    print(state.tasks[0].interrupts[0].value["question"])

In [ ]:
# Resume with the user's info as a dict
result = graph_b.invoke(
    Command(resume={"name": "Ahmed", "preferred_language": "Python"}),
    config=config_b
)
print(result["messages"][-1].content)

---
# Pattern C — Human Reviews and Edits Draft Output

**Use case:** Agent writes a draft. Human reviews, approves or gives feedback.  
Agent revises until approved (max 3 rounds).

```
START → write_draft → review ←interrupt→ "approve" → END
              ↑                          "feedback" ↓
              └──────────── revise ←────────────────┘
```

In [ ]:
class StateC(TypedDict):
    messages:       Annotated[list, add_messages]
    draft:          str
    approved:       bool
    revision_count: int


writer_llm = ChatOllama(model="llama3.2", temperature=0.5)


def write_draft(state: StateC) -> dict:
    resp = writer_llm.invoke([SystemMessage(content="You are a professional writer. Be concise.")] + state["messages"])
    rev = state["revision_count"] + 1
    print(f"\n📝 Draft #{rev} written")
    return {"draft": resp.content, "messages": [resp], "revision_count": rev}


def review_node(state: StateC) -> dict:
    print(f"\n{'='*50}")
    print(f"DRAFT (v{state['revision_count']}):")
    print(state["draft"])
    print('='*50)
    feedback = interrupt({"question": "Type 'approve' or give feedback:", "draft": state["draft"]})
    if str(feedback).lower() == "approve":
        return {"approved": True}
    return {"approved": False, "messages": [HumanMessage(content=f"Revise: {feedback}")]}


b_c = StateGraph(StateC)
b_c.add_node("write",  write_draft)
b_c.add_node("review", review_node)
b_c.add_edge(START,    "write")
b_c.add_edge("write",  "review")
b_c.add_conditional_edges("review",
    lambda s: "end" if (s.get("approved") or s.get("revision_count", 0) >= 3) else "write",
    {"write": "write", "end": END})

ckpt_c  = MemorySaver()
graph_c = b_c.compile(checkpointer=ckpt_c)
print("Graph C compiled!")

In [ ]:
config_c = {"configurable": {"thread_id": "hitl-c-1"}}

# Run — graph writes draft and pauses for review
graph_c.invoke(
    {"messages": [HumanMessage(content="Write a 2-sentence job posting for a Python developer.")],
     "draft": "", "approved": False, "revision_count": 0},
    config=config_c
)

state = graph_c.get_state(config_c)
if state.tasks and state.tasks[0].interrupts:
    print("\n" + state.tasks[0].interrupts[0].value["question"])

In [ ]:
# Option 1: Approve as-is
result = graph_c.invoke(Command(resume="approve"), config=config_c)
print("Final approved draft:")
print(result["draft"])

In [ ]:
# Option 2 (try in a new thread): Request revision
config_c2 = {"configurable": {"thread_id": "hitl-c-2"}}
graph_c.invoke(
    {"messages": [HumanMessage(content="Write a 2-sentence job posting for a Python developer.")],
     "draft": "", "approved": False, "revision_count": 0},
    config=config_c2
)

# Request a revision instead of approving
result2 = graph_c.invoke(Command(resume="Make it more exciting and mention remote work"), config=config_c2)

# Then approve the revised version
state2 = graph_c.get_state(config_c2)
if state2.next:
    result2 = graph_c.invoke(Command(resume="approve"), config=config_c2)
print("\nFinal after revision:")
print(result2.get("draft", ""))

## Key Takeaways

| Concept | Description |
|---------|-------------|
| `interrupt(value)` | Pauses graph, returns `value` to caller |
| `MemorySaver` | Required — saves state so graph can be resumed |
| `thread_id` | Identifies which session to resume |
| `Command(resume=x)` | Resumes the graph with `x` as the interrupt's return value |
| `graph.get_state(config)` | Inspect current state and pending interrupts |
| `state.next` | Non-empty when graph is paused |

## 🏋️ Exercise
Build a **budget approval workflow**:
1. Agent proposes a purchase (item + cost)
2. If cost > $500 → interrupt for human approval
3. If cost ≤ $500 → auto-approve (no interrupt)
4. Test: `"Buy a monitor for $450"` and `"Buy a server rack for $8000"`

In [ ]:
# Your budget approval exercise here
